# 音声研究のための統計解析 〜RとColaboratoryで学ぶ実践ハンドブック〜

### 第9章：多変量モデル・関数データ解析：測定誤差モデルの展望


ここに掲載されているコードは、
『音声研究のための統計解析 〜RとColaboratoryで学ぶ実践ハンドブック〜』の付録である。
本書の巻末に載っているGitHubリポジトリのリンクから、「Open in Colab」のバッジで開いている。

「このノートブックはGoogleが作成したものではありません」という警告は、外部（GitHub）から
読み込むノートブック全般に出る定型の警告である。内容を確認のうえ実行してよい。
書き込んだ内容を残したいときは、「ファイル」→「ドライブにコピーを保存」で自分のGoogle Driveに
保存する。保存したコピーは自由に変更・実行・保存してよい。

**説明はすべてコード中の#コメントとセル内のmarkdownに書いてある。**
コピペ元を別に参照する必要はなく、このノートブックを上から実行しながら読めば完結する。

実行前に、Colabメニューの「ランタイム」→「ランタイムのタイプを変更」で言語を **R** に切り替えること。
既定のPythonランタイムのままだと以降のセルがすべてエラーになる。


参照論文: Hu, Bürkner & Arvaniti (2026), *Journal of Speech, Language, and Hearing Research*／Ramsay & Silverman (2005), *Functional Data Analysis*。

In [ ]:
# ⏳ brmsのインストールは数分かかることがある。
install.packages(c("brms", "refund"), repos = "https://cloud.r-project.org")
library(brms)
library(tidyverse)


In [ ]:
# ---- 共通データセットの読み込み ----
# 第1章と全く同じ手順・同じ乱数seedでVOTデータを作り直す。
# 「本ごとに違うデータ」を避けるためだが、外部URLから読み込むのではなく
# その場で再生成する設計にしてある——理由は、配布用ノートブックが
# 非公開のGitHubリポジトリのURLに依存すると、読者の環境からは
# アクセスできず404になってしまうため（実際にこの問題が起きて修正した）。
# コードを自分の環境で動かす分には、この方式のほうが「ネットワーク不要で
# 必ず同じ結果になる」という利点もある。
if (!requireNamespace("tidyverse", quietly = TRUE)) install.packages("tidyverse", repos = "https://cloud.r-project.org")
library(tidyverse)
# ---- 図の日本語文字化け対策・文字サイズの既定値 ----
# Colabの既定フォントには日本語グリフが含まれないため、図のタイトルや軸ラベルが
# 文字化け（豆腐記号）して表示される。Google Fontsの日本語フォントを取得し、
# "notosans"という名前で登録して使う（要ネット接続。Colabでは常に使える）。
if (!requireNamespace("showtext", quietly = TRUE)) install.packages("showtext", repos = "https://cloud.r-project.org")
library(showtext)
font_add_google("Noto Sans JP", "notosans")  # "sans"などの予約名は上書きできないため専用名で登録
showtext_auto()
showtext_opts(dpi = 100)                     # Colabのインライン画像解像度(100dpi)に合わせる
par(family = "notosans")                     # base Rの作図（mclust・itsadugなど）にも適用する
theme_set(theme_minimal(base_size = 14, base_family = "notosans"))  # ggplotの既定テーマ


set.seed(42)  # 乱数のseedを固定する。同じ数字なら誰が実行しても同じ結果になる
n_speaker <- 10                    # 話者数
n_word    <- 20                    # 単語数（各話者が全単語を1回ずつ発話する）
n_obs     <- n_speaker * n_word    # = 200行

# 何行目がどの話者・どの単語かを表す添字。あとで話者ごと・単語ごとの
# 値を引くのに使う（speaker_intercept[speaker_idx] のように書ける）。
speaker_idx <- rep(1:n_speaker, each  = n_word)
word_idx    <- rep(1:n_word,    times = n_speaker)

# 1行ごとにvoiced/voicelessを交互に並べる。話者1人あたり20行なので、
# どの話者を見ても必ずvoiced10回・voiceless10回が含まれる
# ——これは第5章で「話者ごとにconditionの効果を推定する」ために必須の設計。
condition_vec <- rep(c("voiced", "voiceless"), times = n_obs / 2)

# ---- 話者差・単語差を「先に」作っておく ----
# 実際の音声データでは、同じ条件でも話者によってVOTの平均が違い（声道長・
# 発話速度・方言）、単語によっても違う（後続母音・語長・頻度）。
# 擬似データでもこの構造を明示的に作っておかないと、第5章以降で使う
# 混合効果モデルが「推定すべきばらつきが最初から存在しない」データを
# 相手にすることになり、ランダム効果の分散推定値が0（singular fit）になってしまう。
speaker_intercept <- rnorm(n_speaker, mean = 0, sd = 6)
# 話者ごとのVOTの底上げ量。どの条件にも一律にかかる（＝ランダム切片）。
speaker_slope <- rnorm(n_speaker, mean = 0, sd = 10)
# 話者ごとの「有声と無声をどれだけ大きく作り分けるか」の差（＝ランダム傾き）。
# 話者間のばらつきは無声（VOTが長い側）で特に大きいことが知られているので、
# voiceless条件にだけ加える。
word_intercept <- rnorm(n_word, mean = 0, sd = 3)
# 単語ごとの癖（＝項目のランダム切片）。

vot_data <- tibble(
  speaker   = paste0("S", sprintf("%02d", speaker_idx)),
  word      = paste0("word_", word_idx),
  gender    = rep(rep(c("M", "F"), each = n_speaker / 2), each = n_word),
  condition = condition_vec,
  # VOTは「固定効果 + 話者差 + 単語差 + 残差」を足し合わせて作る。
  # これは第5章で当てはめる混合効果モデルの式そのものであり、
  # モデルが推定した分散をこの生成時の値と見比べられるようにしてある。
  VOT = 20 +                                              # 有声条件の基準値
    if_else(condition == "voiceless", 55, 0) +            # 無声条件の上乗せ（→約75ms）
    speaker_intercept[speaker_idx] +                      # 話者のランダム切片
    if_else(condition == "voiceless",                     # 話者のランダム傾き
            speaker_slope[speaker_idx], 0) +
    word_intercept[word_idx] +                            # 単語のランダム切片
    # 残差。無声条件のほうがばらつきが大きい（実際のVOTデータもそうなっている）。
    # 第3章のモデル診断で「等分散性が崩れている」例として使う。
    rnorm(n_obs, mean = 0, sd = if_else(condition == "voiced", 4, 8)),
  F0 = rnorm(n_obs, mean = 130, sd = 15)  # F0はここでは条件と無関係に生成
)

vot_data <- vot_data %>%
  mutate(
    # 文字列のままだとRが「順序のない値」として扱うため、
    # 話者・単語・性別・条件をすべてfactor（因子）型に変換しておく。
    # 特にconditionはlevels=でvoiced/voicelessの順序を明示し、
    # 以降のモデルの切片が「voiced群の平均」になるよう固定する。
    speaker   = factor(speaker),
    word      = factor(word),
    gender    = factor(gender),
    condition = factor(condition, levels = c("voiced", "voiceless"))
  )

glimpse(vot_data)  # 型と最初の数行を確認。数値であるべき列が<chr>のままなら要注意

# --- 実行結果 ---
# Rows: 200
# Columns: 6
# $ speaker   <fct> S01, S01, S01, S01, S01, S01, S01, S01, S01, S01, S01, ...
# $ word      <fct> word_1, word_2, word_3, word_4, word_5, word_6, word_7, ...
# $ gender    <fct> M, M, M, M, M, M, M, M, M, M, M, M, M, M, M, M, M, M, ...
# $ condition <fct> voiced, voiceless, voiced, voiceless, voiced, voiceless, ...
# $ VOT       <dbl> 28.129829, 88.042064, 30.742652, 94.104833, 28.438207, ...
# $ F0        <dbl> 119.0617, 144.9710, 148.8772, 148.7330, 109.2904, ...
#
# 【出力の読み方】
# ・Rows: 200 / Columns: 6 が行数と列数。10話者×20語=200行になっているかを
#   まず確認する。ここが合っていなければ、以降の分析はすべて無意味になる。
# ・各行の <fct> <dbl> が**型**である。<fct>はfactor（水準を持つカテゴリ）、
#   <dbl>は小数を含む数値。speaker・word・gender・conditionがすべて<fct>、
#   VOTとF0が<dbl>になっていれば正しい。
#   **VOTが<chr>（文字列）になっていたら、そこで止まって読み込みを見直す。**
#   文字列のままでは平均もモデルも計算できず、しかもエラーではなく
#   「変な結果」として静かに進んでしまうことがある。
# ・conditionの並びがvoiced, voiceless, voiced, voiceless...と1行おきに
#   交互になっている。これは意図した設計どおり（どの話者も両条件を10回ずつ持つ）。
# ・speakerが先頭でS01ばかり並ぶのは、話者ごとに20行がまとまっているため。
#   行の並び順自体は分析結果に影響しない（第7章のAR(1)を使うときだけ例外）。


■ 多変量ベイズモデル：VOTとF0を同時にモデル化する：

**実際の研究例**：有声性の対立は多くの言語でVOTだけでなくF0（onset f0）にも同時に現れることが知られている。これらを別々のモデルで解析すると、「話者Aは全体的にVOTが長くF0も高い」といった話者レベルでの指標間の相関を捉えられない。brmsのマルチバリエートモデルのチュートリアル群が示す方向性にならい、VOTとF0を別々に解析する代わりに`mvbind()`で同時にモデル化し、話者レベルの相関を推定する。

**ただし本書のデータには、この相関が入っていない。** F0は`rnorm(n_obs, mean = 130, sd = 15)`で条件・話者と無関係に生成してあり（第1章参照）、VOTの話者差とは独立である。つまりこの節は「既知の正の相関を正しく検出できるか」ではなく、**「相関が無いデータに多変量モデルを当てても、無いものを作り出さないか」**を確認する節である。実データでこの構造の相関を検出したい場合は、データ生成側でF0にも話者レベルの効果を意図的に入れたシミュレーションで、まず手法が機能することを確認してから使うとよい。

In [ ]:
# ⏳ このセルも1〜3分ほどかかる（Stanコンパイル＋4鎖サンプリング）。
model_mv <- brm(
  # mvbind(VOT, F0): 2つの応答変数を1つのモデルで同時に予測する（多変量モデル）
  # (1 + condition | p | speaker): "p |" が話者ごとのランダム効果の相関を
  #   VOTとF0の間で共有させる部分。これにより「VOTが長い話者はF0も高いか」
  #   という話者レベルの相関を直接推定できる。
  # (1 | word): ★単語のランダム切片も、第5章以降と同じ理由で必ず入れる。
  #   本書のデータは同じ話者から複数の単語を、同じ単語を複数の話者から
  #   取っている（交差デザイン）。多変量モデルだからといって、この反復構造を
  #   無視してよい理由にはならない。
  mvbind(VOT, F0) ~ condition + (1 + condition | p | speaker) + (1 | word),
  data = vot_data,
  chains = 4, iter = 2000, warmup = 1000, seed = 42,
  control = list(adapt_delta = 0.95)
)

# 収束診断は複雑なモデルほど省略しないこと（第8章参照）。
cat("max Rhat    :", round(max(brms::rhat(model_mv)), 4), "\n")
np <- brms::nuts_params(model_mv)
cat("divergences :", sum(subset(np, Parameter == "divergent__")$Value), "\n")

summary(model_mv)

# --- 実行結果（要約） ---
# max Rhat    : 1.0043
# divergences : 0
# （収束は良好。Rhatが1.01を超えるパラメータ・発散遷移とも無し）


■ 話者レベルの相関を読む：

`p |` でランダム効果の相関行列を共有させると、"VOTが長い話者ほどF0も高い/低い"
といった指標間の関係を推定できる（単変量モデルを2本別々に走らせては得られない）。

In [ ]:
vc <- VarCorr(model_mv)$speaker$cor
cat("話者レベルの切片どうしの相関（VOTの切片 x F0の切片）:\n")
cat("  推定値:", round(vc["VOT_Intercept", "Estimate", "F0_Intercept"], 3),
    " [", round(vc["VOT_Intercept", "Q2.5", "F0_Intercept"], 3), ",",
    round(vc["VOT_Intercept", "Q97.5", "F0_Intercept"], 3), "]\n")

cat("残差レベルの相関（rescor、話者・単語の効果を除いた後）:\n")
print(round(summary(model_mv)$rescor_pars, 3))

# --- 実行結果 ---
# 話者レベルの切片どうしの相関（VOTの切片 x F0の切片）:
#   推定値: -0.19  [ -0.878 , 0.706 ]
# 残差レベルの相関（rescor、話者・単語の効果を除いた後）:
#                 Estimate Est.Error l-95% CI u-95% CI  Rhat Bulk_ESS Tail_ESS
# rescor(VOT,F0)    -0.096     0.075   -0.241    0.055 1.001     4541     3337
#
# 【出力の読み方】
# ・**話者レベルの相関の95%信用区間は−0.878から0.706と、0付近の広い範囲を
#   覆っている。** これは「相関が無い」ことの積極的な証明ではないが、
#   「正の相関がある」という主張も「負の相関がある」という主張も、
#   このデータからは支持されない。前段で述べたとおり、本書のデータは
#   VOTとF0を独立に生成してあるので、これは期待どおりの結果である。
# ・残差レベルの相関（rescor）も−0.096で、信用区間（−0.241〜0.055）が
#   0を含む。話者・単語の効果を除いた後の、試行ごとのばらつきの相関も
#   検出されていない。
# ・**この「見つからなかった」という結果自体が、モデルの妥当性の確認になる。**
#   相関の無いデータに多変量モデルを当てて相関を作り出してしまうようでは、
#   実データで相関が見つかったときにそれを信用できない。まずこの確認を
#   経てから、実データでの解釈に進む。


■ 平均が同じでも分布は違う：

平均は同じで、ばらつき（SD）だけが条件で違う人工データを作る。t検定やlmは平均の差しか検定しないので「差なし」と答えるが、glmmTMBの dispformula で分散の式を別に立てると、ばらつきの条件差が推定できる。出力のあとの【出力の読み方】を確認しよう。

In [ ]:
if (!requireNamespace("glmmTMB", quietly = TRUE)) install.packages("glmmTMB", repos = "https://cloud.r-project.org")
library(glmmTMB)
set.seed(2026)
d <- data.frame(
  cond = factor(rep(c("A", "B"), each = 60)),
  y    = c(rnorm(60, mean = 50, sd = 5), rnorm(60, mean = 50, sd = 15))
)
# (a) 平均の差：差は出ない
print(t.test(y ~ cond, data = d, var.equal = TRUE))
# (b) 条件ごとの平均とSD
print(aggregate(y ~ cond, data = d, FUN = function(v) c(mean = mean(v), sd = sd(v))))
# (c) 分散の条件差を推定
m <- glmmTMB(y ~ cond, dispformula = ~ cond, data = d)
print(summary(m))

# --- 実行結果 ---
# (a) t = 0.41229, df = 118, p-value = 0.6809 （平均の差は検出されない）
# (b) A: 平均49.449 / SD 4.711   B: 平均48.540 / SD 16.420
# (c) 平均部分 condB = -0.9092 (SE 2.1868, p = 0.678)
#     分散部分 condB =  1.2486 (SE 0.1291, z = 9.672, p < 2e-16)
#     分散部分の係数はSDの対数スケール：SD比 = exp(1.2486) ≒ 3.5倍
# 【出力の読み方】平均は条件間でほぼ同じ(p=0.68)なのに、SDは約3.5倍違い、
# 分散の式を別に立てたときだけその差が有意に出る。平均だけを見ていると見逃す差である。


■ 分布パラメータそのものをモデル化する（distributional model）：

平均だけでなく分散（sigma）も予測変数の関数にする。

In [ ]:
# ⏳ 1〜3分ほどかかる。
model_dist <- brm(
  # bf(): brms formula。sigma ~ conditionにより「ばらつきの大きさ」自体を
  # conditionの関数にする。平均だけでなく分散にも群間差があるかを検定できる。
  # ★平均の式には、他の章と同じ話者・単語のランダム効果を必ず入れる。
  #   distributional modelを使うことと、反復測定の構造を無視してよいことは
  #   別問題である。
  bf(VOT ~ condition + (1 + condition | speaker) + (1 | word),
     sigma ~ condition),
  data = vot_data,
  chains = 4, iter = 2000, warmup = 1000, seed = 42,
  control = list(adapt_delta = 0.95)
)

cat("max Rhat    :", round(max(brms::rhat(model_dist)), 4), "\n")
np <- brms::nuts_params(model_dist)
cat("divergences :", sum(subset(np, Parameter == "divergent__")$Value), "\n")

summary(model_dist)
pp_check(model_dist, ndraws = 100)  # 事後予測チェック。実データの分布とシミュレーションを重ねる

# --- 実行結果（要約） ---
# max Rhat    : 1.0066
# divergences : 0
#
# 固定効果:
#                            Estimate Est.Error  l-95% CI  u-95% CI
# Intercept                    22.771     2.168    18.359    27.023
# conditionvoiceless           52.307     6.760    38.909    65.739
# sigma_Intercept                1.291     0.078     1.145     1.446
# sigma_conditionvoiceless       0.749     0.110     0.539     0.963
#
# 【出力の読み方】
# ・sigma側の係数はlogスケールで返る。有声条件の残差SDは exp(1.291) ≈ 3.6ms、
#   無声条件は exp(1.291 + 0.749) ≈ 7.7ms と読む。生成時点の残差SD（有声4ms、
#   無声8ms）に近い値が回収できている。
# ・sigma_conditionvoicelessの95%区間（0.539〜0.963）が0を明確に上回っており、
#   「無声条件のほうがばらつきが大きい」という第3章以来の観察を、話者・単語の
#   反復構造を考慮したうえで確認できたことになる。


■ 測定誤差モデルの考え方：

自動計測（forced alignment等）のVOTには測定誤差が乗る。brmsでは
`mi()`を使って測定誤差を明示的にモデルへ組み込める。

In [ ]:
vot_data_me <- vot_data %>%
  mutate(VOT_se = 2)  # 測定誤差のSDを仮に2msとする（実データでは推定手法の精度から見積もる）

# ⏳ 1〜3分ほどかかる。
model_me <- brm(
  # mi(VOT_se): VOTの測定に誤差があることを明示し、真値の不確実性を伝播させる。
  # ★測定誤差モデルでも、話者・単語のランダム効果は変わらず必要である。
  VOT | mi(VOT_se) ~ condition + (1 + condition | speaker) + (1 | word),
  data = vot_data_me,
  chains = 4, iter = 2000, warmup = 1000, seed = 42,
  control = list(adapt_delta = 0.95)
)

cat("max Rhat    :", round(max(brms::rhat(model_me)), 4), "\n")
np <- brms::nuts_params(model_me)
cat("divergences :", sum(subset(np, Parameter == "divergent__")$Value), "\n")

summary(model_me)

# --- 実行結果（要約） ---
# max Rhat    : 1.0038
# divergences : 1
#
# 固定効果:
#                     Estimate Est.Error l-95% CI u-95% CI
# Intercept             22.821     2.100   18.888   26.946
# conditionvoiceless    52.142     7.003   38.397   65.871
#
# 【出力の読み方】
# ・**発散遷移が1件出ている。** 第8章で述べたとおり、これを無視して結果だけ
#   報告してはいけない。1件だけであれば多くの場合実害は小さいが、
#   `adapt_delta`をさらに上げて（例: 0.99）再実行し、発散が消えるか、
#   係数の推定値が動かないかを確認するのが望ましい対応である。ここでは
#   紙面の都合で1回の実行結果をそのまま示すが、実際の研究では発散が
#   0になるまで調整してから結果を確定させること。
# ・係数の値自体は測定誤差を考慮しない通常のモデル（第5章）と大きくは
#   違わない。これは、ここで仮定した測定誤差（SD 2ms）がVOTの効果の
#   大きさ（52ms）に比べて小さいためである。測定誤差の影響は、誤差が
#   効果量に対して相対的に大きいときほど顕著になる。


■ 関数データ解析（FDA）への橋渡し：

第7章のGAMMは「時間×条件」の平均的な軌跡を推定するが、FDAは1発話ごとのF0曲線を1つの関数的観測として扱い、functional PCAで主要な変動パターンを取り出す（Ramsay & Silverman, 2005）。`refund::fpca.sc()`などが対応する。

本書では概念の紹介にとどめ、実装は発展課題とする。

■ 演習：

1. `model_mv`の話者レベルの相関（残差の相関`rescor`とは別物）から、VOTとF0の話者レベルのトレードオフがあるかを議論せよ。
2. `model_dist`で条件によって分散（ばらつき）が変わるかを確認し、平均だけを見ていては分からない結果があるか考察せよ。